# TraderPro - RL Training V2 (DQN + A2C)

Version améliorée avec:
- **DQN** (Deep Q-Network) - Meilleur pour actions discrètes
- **A2C** (Advantage Actor-Critic) - Plus rapide que PPO
- **Episodes courts** - Max 60 jours par épisode
- **Observations normalisées** - Stabilité d'entraînement
- **Reward shaping amélioré** - Signal plus clair

## Pourquoi PPO bloquait ?
1. Episodes trop longs (800+ jours) → gradient vanishing
2. Observations non normalisées → instabilité
3. Reward trop sparse → pas d'apprentissage

---
## 1. Installation

In [ ]:
!pip install stable-baselines3[extra] gymnasium pandas numpy requests ta matplotlib --quiet
print("Installation terminée!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Reinforcement Learning
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy

print("Imports OK!")
print(f"Gymnasium version: {gym.__version__}")

---
## 2. Récupération des données

In [ ]:
# Configuration Twelve Data API
TWELVEDATA_API_KEY = "7521bc3c3e92455181d590829c90bf9e"

def fetch_twelvedata(symbol="BTC/USD", interval="1day", years=3):
    """
    Récupère les données historiques depuis Twelve Data API.
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years)
    
    url = "https://api.twelvedata.com/time_series"
    
    params = {
        "symbol": symbol,
        "interval": interval,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "order": "asc",
        "apikey": TWELVEDATA_API_KEY,
        "outputsize": 5000
    }
    
    print(f"Récupération des données {symbol}...")
    print(f"Période: {start_date.strftime('%Y-%m-%d')} à {end_date.strftime('%Y-%m-%d')}")
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if "code" in data and data["code"] != 200:
        raise Exception(f"Erreur API: {data.get('message', 'Unknown error')}")
    
    if "values" not in data:
        raise Exception(f"Pas de données reçues. Réponse: {data}")
    
    df = pd.DataFrame(data["values"])
    df = df.rename(columns={"datetime": "timestamp"})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    
    df = df[['open', 'high', 'low', 'close', 'volume']]
    df = df.sort_index()
    
    return df

# Récupérer les données
df = fetch_twelvedata("BTC/USD", "1day", years=3)
print(f"\nDonnées récupérées: {len(df)} jours")
print(f"Période: {df.index[0]} à {df.index[-1]}")
df.tail()

---
## 3. Feature Engineering (Amélioré)

In [ ]:
def create_features(df):
    """
    Crée les features normalisées pour l'agent RL.
    Toutes les features sont normalisées entre -1 et 1 ou 0 et 1.
    """
    data = df.copy()
    
    # Returns (déjà petits, pas besoin de normaliser beaucoup)
    data['returns'] = data['close'].pct_change().clip(-0.2, 0.2)  # Clip à ±20%
    data['returns_5d'] = data['close'].pct_change(5).clip(-0.5, 0.5)
    data['returns_10d'] = data['close'].pct_change(10).clip(-0.5, 0.5)
    
    # Volatilité normalisée
    data['volatility_7d'] = data['returns'].rolling(7).std()
    vol_mean = data['volatility_7d'].expanding().mean()
    vol_std = data['volatility_7d'].expanding().std()
    data['volatility_norm'] = ((data['volatility_7d'] - vol_mean) / (vol_std + 1e-8)).clip(-3, 3) / 3
    
    # RSI normalisé entre -1 et 1
    delta = data['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-10)
    data['rsi'] = 100 - (100 / (1 + rs))
    data['rsi_norm'] = (data['rsi'] - 50) / 50  # Entre -1 et 1
    
    # MACD normalisé
    ema12 = data['close'].ewm(span=12).mean()
    ema26 = data['close'].ewm(span=26).mean()
    data['macd'] = ema12 - ema26
    data['macd_signal'] = data['macd'].ewm(span=9).mean()
    data['macd_hist'] = data['macd'] - data['macd_signal']
    # Normaliser par le prix
    data['macd_norm'] = (data['macd_hist'] / data['close'] * 100).clip(-5, 5) / 5
    
    # Bollinger Bands position (0 à 1)
    sma20 = data['close'].rolling(20).mean()
    std20 = data['close'].rolling(20).std()
    data['bb_upper'] = sma20 + 2 * std20
    data['bb_lower'] = sma20 - 2 * std20
    data['bb_position'] = ((data['close'] - data['bb_lower']) / (data['bb_upper'] - data['bb_lower'] + 1e-10)).clip(0, 1)
    # Transformer en -1 à 1
    data['bb_norm'] = data['bb_position'] * 2 - 1
    
    # Trend (prix vs SMA)
    data['sma_20'] = data['close'].rolling(20).mean()
    data['sma_50'] = data['close'].rolling(50).mean()
    data['trend_short'] = ((data['close'] / data['sma_20']) - 1).clip(-0.3, 0.3) / 0.3
    data['trend_long'] = ((data['close'] / data['sma_50']) - 1).clip(-0.5, 0.5) / 0.5
    
    # Volume normalisé
    data['volume_sma'] = data['volume'].rolling(20).mean()
    data['volume_ratio'] = (data['volume'] / (data['volume_sma'] + 1e-10)).clip(0, 5) / 5 * 2 - 1
    
    # Momentum
    data['momentum'] = data['returns'].rolling(10).mean().clip(-0.1, 0.1) / 0.1
    
    # Clean
    data = data.replace([np.inf, -np.inf], 0)
    data = data.fillna(0)
    
    return data

# Créer les features
df_features = create_features(df)

# Vérifier la normalisation
feature_cols = ['returns', 'returns_5d', 'returns_10d', 'volatility_norm', 
                'rsi_norm', 'macd_norm', 'bb_norm', 'trend_short', 'trend_long',
                'volume_ratio', 'momentum']

print("Vérification de la normalisation des features:")
print("="*50)
for col in feature_cols:
    print(f"{col:20s}: min={df_features[col].min():6.2f}, max={df_features[col].max():6.2f}")

print(f"\nNombre de lignes: {len(df_features)}")

---
## 4. Environnement de Trading V2 (Amélioré)

In [ ]:
class TradingEnvV2(gym.Env):
    """
    Environnement de trading amélioré.
    
    Améliorations:
    - Episodes courts (max_steps configurable)
    - Observations normalisées
    - Reward shaping progressif
    - Démarrage aléatoire pour diversité
    """
    
    metadata = {'render_modes': ['human']}
    
    def __init__(self, df, max_steps=60, initial_capital=10000, 
                 transaction_fee=0.001, random_start=True):
        super().__init__()
        
        self.df = df.reset_index(drop=True)
        self.max_steps = max_steps  # Episodes courts!
        self.initial_capital = initial_capital
        self.transaction_fee = transaction_fee
        self.random_start = random_start
        self.window_size = 50  # Pour avoir assez de données pour les features
        
        # Features normalisées
        self.feature_columns = [
            'returns', 'returns_5d', 'returns_10d', 'volatility_norm',
            'rsi_norm', 'macd_norm', 'bb_norm', 'trend_short', 'trend_long',
            'volume_ratio', 'momentum'
        ]
        
        # Espace d'actions: 0=HOLD, 1=BUY, 2=SELL
        self.action_space = spaces.Discrete(3)
        
        # Observations: features + position + unrealized_pnl + steps_in_position
        n_features = len(self.feature_columns) + 3
        self.observation_space = spaces.Box(
            low=-1.0, 
            high=1.0, 
            shape=(n_features,), 
            dtype=np.float32
        )
        
        self.reset()
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Point de départ aléatoire pour diversité
        max_start = len(self.df) - self.max_steps - self.window_size - 10
        if self.random_start and max_start > self.window_size:
            self.start_idx = np.random.randint(self.window_size, max_start)
        else:
            self.start_idx = self.window_size
        
        self.current_step = 0
        self.data_idx = self.start_idx
        
        self.capital = self.initial_capital
        self.position = 0  # 0 = cash, 1 = long
        self.entry_price = 0
        self.steps_in_position = 0
        
        self.total_profit = 0
        self.trades = []
        self.portfolio_values = [self.initial_capital]
        
        return self._get_observation(), {}
    
    def _get_observation(self):
        """Retourne l'observation normalisée."""
        features = self.df.loc[self.data_idx, self.feature_columns].values.astype(np.float32)
        
        # Position (0 ou 1)
        position_norm = float(self.position)
        
        # PnL non réalisé normalisé
        unrealized_pnl = 0.0
        if self.position == 1:
            current_price = self.df.loc[self.data_idx, 'close']
            unrealized_pnl = (current_price - self.entry_price) / self.entry_price
            unrealized_pnl = np.clip(unrealized_pnl, -0.5, 0.5) * 2  # Normaliser
        
        # Temps en position normalisé (0 à 1)
        steps_norm = min(self.steps_in_position / 30.0, 1.0)
        
        obs = np.concatenate([
            features,
            [position_norm, unrealized_pnl, steps_norm]
        ]).astype(np.float32)
        
        # Assurer que tout est dans [-1, 1]
        obs = np.clip(obs, -1.0, 1.0)
        obs = np.nan_to_num(obs, nan=0.0)
        
        return obs
    
    def step(self, action):
        current_price = self.df.loc[self.data_idx, 'close']
        prev_portfolio = self._get_portfolio_value()
        reward = 0.0
        
        # === EXÉCUTER L'ACTION ===
        if action == 1:  # BUY
            if self.position == 0:
                self.position = 1
                self.entry_price = current_price
                self.capital *= (1 - self.transaction_fee)
                self.steps_in_position = 0
                reward += 0.01  # Petit bonus pour agir
                
        elif action == 2:  # SELL
            if self.position == 1:
                # Calculer le profit
                trade_return = (current_price - self.entry_price) / self.entry_price
                self.capital *= (1 + trade_return) * (1 - self.transaction_fee)
                
                self.trades.append({
                    'return': trade_return,
                    'duration': self.steps_in_position
                })
                
                # Reward basé sur le profit
                reward += trade_return * 10  # Amplifier le signal
                
                # Bonus pour trades gagnants
                if trade_return > 0:
                    reward += 0.5
                else:
                    reward -= 0.2  # Petite pénalité pour pertes
                
                self.position = 0
                self.entry_price = 0
                self.steps_in_position = 0
        
        else:  # HOLD
            if self.position == 1:
                self.steps_in_position += 1
                # Reward pour maintenir une position profitable
                unrealized = (current_price - self.entry_price) / self.entry_price
                if unrealized > 0:
                    reward += unrealized * 0.5  # Encourager à garder les gains
                elif unrealized < -0.05:  # Plus de 5% de perte
                    reward -= 0.1  # Pénalité pour garder une position perdante
        
        # Avancer
        self.current_step += 1
        self.data_idx += 1
        
        # Mettre à jour portfolio
        current_portfolio = self._get_portfolio_value()
        self.portfolio_values.append(current_portfolio)
        
        # Reward basé sur changement de portfolio
        portfolio_return = (current_portfolio - prev_portfolio) / prev_portfolio
        reward += portfolio_return * 5
        
        # === CONDITIONS DE FIN ===
        terminated = False
        truncated = False
        
        # Fin de l'épisode (max steps atteint)
        if self.current_step >= self.max_steps:
            terminated = True
            # Forcer la vente si position ouverte
            if self.position == 1:
                final_price = self.df.loc[self.data_idx - 1, 'close']
                trade_return = (final_price - self.entry_price) / self.entry_price
                self.capital *= (1 + trade_return) * (1 - self.transaction_fee)
                reward += trade_return * 5
        
        # Fin des données
        if self.data_idx >= len(self.df) - 1:
            terminated = True
        
        # Perte catastrophique
        if current_portfolio < self.initial_capital * 0.7:
            truncated = True
            reward -= 2.0  # Grande pénalité
        
        info = {
            'portfolio_value': current_portfolio,
            'position': self.position,
            'trades': len(self.trades),
            'total_return': (current_portfolio - self.initial_capital) / self.initial_capital
        }
        
        return self._get_observation(), reward, terminated, truncated, info
    
    def _get_portfolio_value(self):
        if self.position == 1:
            current_price = self.df.loc[self.data_idx, 'close']
            unrealized = (current_price - self.entry_price) / self.entry_price
            return self.capital * (1 + unrealized)
        return self.capital
    
    def get_metrics(self):
        if len(self.portfolio_values) < 2:
            return {}
        
        returns = np.diff(self.portfolio_values) / np.array(self.portfolio_values[:-1])
        total_return = (self.portfolio_values[-1] - self.initial_capital) / self.initial_capital
        
        sharpe = 0
        if len(returns) > 0 and np.std(returns) > 0:
            sharpe = np.mean(returns) / np.std(returns) * np.sqrt(252)
        
        win_trades = [t for t in self.trades if t['return'] > 0]
        win_rate = len(win_trades) / len(self.trades) if self.trades else 0
        
        return {
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'total_trades': len(self.trades),
            'win_rate': win_rate,
            'final_capital': self.portfolio_values[-1]
        }


# Test de l'environnement
print("Test de l'environnement V2...")
test_env = TradingEnvV2(df_features, max_steps=60)
obs, _ = test_env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Observation range: [{obs.min():.2f}, {obs.max():.2f}]")
print(f"Action space: {test_env.action_space}")

# Test quelques steps
total_reward = 0
for i in range(10):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
    total_reward += reward
    print(f"Step {i+1}: Action={['HOLD','BUY','SELL'][action]}, Reward={reward:.4f}, Portfolio=${info['portfolio_value']:.2f}")

print(f"\nTotal reward (10 steps): {total_reward:.4f}")

---
## 5. Préparation des données Train/Test

In [ ]:
# Split temporel 80/20
split_idx = int(len(df_features) * 0.8)
train_df = df_features.iloc[:split_idx].reset_index(drop=True)
test_df = df_features.iloc[split_idx:].reset_index(drop=True)

print(f"Train: {len(train_df)} jours")
print(f"Test: {len(test_df)} jours")

# Créer les environnements
def make_env(df, max_steps=60, random_start=True):
    def _init():
        env = TradingEnvV2(df, max_steps=max_steps, random_start=random_start)
        env = Monitor(env)
        return env
    return _init

# Environnements
train_env = DummyVecEnv([make_env(train_df, max_steps=60, random_start=True)])
eval_env = DummyVecEnv([make_env(test_df, max_steps=60, random_start=False)])

print("Environnements créés!")

---
## 6. Callback d'entraînement

In [ ]:
class ProgressCallback(BaseCallback):
    """
    Callback pour suivre l'entraînement et sauvegarder le meilleur modèle.
    """
    def __init__(self, eval_env, eval_freq=2000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.best_return = -np.inf
        self.history = []
    
    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            # Évaluer sur plusieurs épisodes
            returns = []
            for _ in range(5):
                obs = self.eval_env.reset()
                done = False
                episode_reward = 0
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, done, info = self.eval_env.step(action)
                    episode_reward += reward[0]
                returns.append(info[0].get('total_return', 0))
            
            avg_return = np.mean(returns)
            self.history.append({'step': self.n_calls, 'return': avg_return})
            
            if self.verbose > 0:
                print(f"\n[Step {self.n_calls}] Avg Return: {avg_return*100:.2f}%")
            
            if avg_return > self.best_return:
                self.best_return = avg_return
                self.model.save("best_model")
                if self.verbose > 0:
                    print(f"  → Nouveau meilleur modèle! Return: {avg_return*100:.2f}%")
        
        return True

---
## 7. Entraînement DQN (Recommandé pour actions discrètes)

In [ ]:
# Configuration DQN
print("="*60)
print("ENTRAÎNEMENT DQN")
print("="*60)
print("DQN est optimisé pour les actions discrètes (HOLD/BUY/SELL)")
print()

# Créer le modèle DQN
model_dqn = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=1e-4,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=64,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1000,
    exploration_fraction=0.3,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    verbose=1,
    tensorboard_log="./logs_dqn/"
)

print("Modèle DQN créé!")
print(f"Policy: {model_dqn.policy}")

In [ ]:
# Entraînement DQN
TIMESTEPS_DQN = 50000  # Plus rapide que PPO

callback_dqn = ProgressCallback(eval_env, eval_freq=5000, verbose=1)

print(f"Début entraînement DQN ({TIMESTEPS_DQN} timesteps)...")
print("="*60)

model_dqn.learn(
    total_timesteps=TIMESTEPS_DQN,
    callback=callback_dqn,
    progress_bar=True
)

print("="*60)
print("Entraînement DQN terminé!")
print(f"Meilleur return: {callback_dqn.best_return*100:.2f}%")

---
## 8. Entraînement A2C (Alternative rapide)

In [ ]:
# Recréer les environnements
train_env_a2c = DummyVecEnv([make_env(train_df, max_steps=60, random_start=True)])
eval_env_a2c = DummyVecEnv([make_env(test_df, max_steps=60, random_start=False)])

print("="*60)
print("ENTRAÎNEMENT A2C")
print("="*60)
print("A2C est plus rapide que PPO et souvent plus stable")
print()

# Créer le modèle A2C
model_a2c = A2C(
    "MlpPolicy",
    train_env_a2c,
    learning_rate=7e-4,
    n_steps=16,  # Court pour updates fréquents
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    normalize_advantage=True,
    verbose=1,
    tensorboard_log="./logs_a2c/"
)

print("Modèle A2C créé!")

In [ ]:
# Entraînement A2C
TIMESTEPS_A2C = 50000

callback_a2c = ProgressCallback(eval_env_a2c, eval_freq=5000, verbose=1)

print(f"Début entraînement A2C ({TIMESTEPS_A2C} timesteps)...")
print("="*60)

model_a2c.learn(
    total_timesteps=TIMESTEPS_A2C,
    callback=callback_a2c,
    progress_bar=True
)

print("="*60)
print("Entraînement A2C terminé!")
print(f"Meilleur return: {callback_a2c.best_return*100:.2f}%")

---
## 9. Comparaison DQN vs A2C

In [ ]:
def evaluate_model(model, env_df, model_name, n_episodes=10):
    """
    Évalue un modèle sur plusieurs épisodes.
    """
    env = TradingEnvV2(env_df, max_steps=len(env_df)-100, random_start=False)
    
    results = []
    all_actions = []
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        episode_actions = []
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            episode_actions.append(['HOLD', 'BUY', 'SELL'][action])
        
        metrics = env.get_metrics()
        results.append(metrics)
        all_actions.extend(episode_actions)
    
    # Moyennes
    avg_return = np.mean([r['total_return'] for r in results])
    avg_sharpe = np.mean([r['sharpe_ratio'] for r in results])
    avg_trades = np.mean([r['total_trades'] for r in results])
    avg_winrate = np.mean([r['win_rate'] for r in results])
    
    # Distribution des actions
    action_dist = pd.Series(all_actions).value_counts(normalize=True)
    
    print(f"\n{'='*50}")
    print(f"RÉSULTATS {model_name}")
    print(f"{'='*50}")
    print(f"Return moyen:     {avg_return*100:+.2f}%")
    print(f"Sharpe Ratio:     {avg_sharpe:.2f}")
    print(f"Trades moyens:    {avg_trades:.1f}")
    print(f"Win Rate:         {avg_winrate*100:.1f}%")
    print(f"\nDistribution des actions:")
    for action, pct in action_dist.items():
        print(f"  {action}: {pct*100:.1f}%")
    
    return {
        'model_name': model_name,
        'avg_return': avg_return,
        'avg_sharpe': avg_sharpe,
        'avg_trades': avg_trades,
        'avg_winrate': avg_winrate,
        'action_dist': action_dist
    }

# Charger les meilleurs modèles
try:
    best_dqn = DQN.load("best_model")
    print("Meilleur modèle chargé (dernier sauvegardé)")
except:
    best_dqn = model_dqn
    print("Utilisation du modèle DQN final")

# Évaluer les deux modèles
results_dqn = evaluate_model(model_dqn, test_df, "DQN", n_episodes=5)
results_a2c = evaluate_model(model_a2c, test_df, "A2C", n_episodes=5)

In [ ]:
# Choisir le meilleur modèle
if results_dqn['avg_return'] > results_a2c['avg_return']:
    best_model = model_dqn
    best_name = "DQN"
    best_results = results_dqn
else:
    best_model = model_a2c
    best_name = "A2C"
    best_results = results_a2c

print(f"\n{'='*50}")
print(f"MEILLEUR MODÈLE: {best_name}")
print(f"{'='*50}")
print(f"Return: {best_results['avg_return']*100:+.2f}%")
print(f"Sharpe: {best_results['avg_sharpe']:.2f}")

---
## 10. Backtest Complet du Meilleur Modèle

In [ ]:
# Backtest sur toutes les données de test
backtest_env = TradingEnvV2(test_df, max_steps=len(test_df)-60, random_start=False)
obs, _ = backtest_env.reset()

portfolio_history = [backtest_env.initial_capital]
price_history = [test_df.iloc[backtest_env.start_idx]['close']]
actions_history = []

done = False
while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = backtest_env.step(action)
    done = terminated or truncated
    
    portfolio_history.append(info['portfolio_value'])
    if backtest_env.data_idx < len(test_df):
        price_history.append(test_df.iloc[backtest_env.data_idx]['close'])
    actions_history.append(['HOLD', 'BUY', 'SELL'][action])

# Métriques finales
final_metrics = backtest_env.get_metrics()

print("\n" + "="*60)
print(f"BACKTEST FINAL - {best_name}")
print("="*60)
print(f"Return Total:    {final_metrics['total_return']*100:+.2f}%")
print(f"Sharpe Ratio:    {final_metrics['sharpe_ratio']:.2f}")
print(f"Trades:          {final_metrics['total_trades']}")
print(f"Win Rate:        {final_metrics['win_rate']*100:.1f}%")
print(f"Capital Initial: $10,000")
print(f"Capital Final:   ${final_metrics['final_capital']:,.2f}")

In [ ]:
# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Portfolio vs Buy & Hold
ax1 = axes[0]
initial_price = price_history[0]
buy_hold = [backtest_env.initial_capital * (p / initial_price) for p in price_history]

ax1.plot(portfolio_history[:len(buy_hold)], label=f'Agent {best_name}', color='green', linewidth=2)
ax1.plot(buy_hold, label='Buy & Hold', color='blue', linestyle='--', alpha=0.7)
ax1.axhline(y=backtest_env.initial_capital, color='gray', linestyle=':', alpha=0.5)
ax1.set_title(f'Performance: Agent {best_name} vs Buy & Hold', fontsize=14)
ax1.set_ylabel('Valeur du Portfolio ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Prix BTC avec signaux
ax2 = axes[1]
ax2.plot(price_history, label='Prix BTC', color='orange', alpha=0.7)

for i, action in enumerate(actions_history[:len(price_history)-1]):
    if action == 'BUY':
        ax2.scatter(i, price_history[i], color='green', marker='^', s=50, zorder=5, alpha=0.7)
    elif action == 'SELL':
        ax2.scatter(i, price_history[i], color='red', marker='v', s=50, zorder=5, alpha=0.7)

ax2.set_title('Prix BTC avec Signaux de Trading', fontsize=14)
ax2.set_ylabel('Prix ($)')
ax2.legend(['Prix BTC', 'BUY', 'SELL'])
ax2.grid(True, alpha=0.3)

# 3. Distribution des actions
ax3 = axes[2]
action_counts = pd.Series(actions_history).value_counts()
colors = {'HOLD': 'gray', 'BUY': 'green', 'SELL': 'red'}
bars = ax3.bar(action_counts.index, action_counts.values, 
               color=[colors.get(a, 'blue') for a in action_counts.index])
ax3.set_title('Distribution des Actions', fontsize=14)
ax3.set_ylabel('Nombre')

# Ajouter pourcentages
total = sum(action_counts.values)
for bar, count in zip(bars, action_counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'{count/total*100:.1f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig('rl_backtest_v2.png', dpi=150)
plt.show()

print("\nGraphique sauvegardé: rl_backtest_v2.png")

---
## 11. Sauvegarde pour TraderPro

In [ ]:
import json

# Sauvegarder le meilleur modèle
model_filename = f"trading_agent_{best_name.lower()}_final"
best_model.save(model_filename)

# Configuration
config = {
    "model_type": best_name,
    "model_version": "2.0.0",
    "trained_at": datetime.now().isoformat(),
    "training_timesteps": TIMESTEPS_DQN if best_name == "DQN" else TIMESTEPS_A2C,
    "feature_columns": [
        'returns', 'returns_5d', 'returns_10d', 'volatility_norm',
        'rsi_norm', 'macd_norm', 'bb_norm', 'trend_short', 'trend_long',
        'volume_ratio', 'momentum'
    ],
    "observation_extras": ["position", "unrealized_pnl_norm", "steps_in_position_norm"],
    "actions": ["HOLD", "BUY", "SELL"],
    "action_space": "Discrete(3)",
    "max_episode_steps": 60,
    "initial_capital": 10000,
    "transaction_fee": 0.001,
    "metrics": final_metrics,
    "comparison": {
        "DQN": {
            "return": results_dqn['avg_return'],
            "sharpe": results_dqn['avg_sharpe']
        },
        "A2C": {
            "return": results_a2c['avg_return'],
            "sharpe": results_a2c['avg_sharpe']
        }
    }
}

with open("training_config_rl_v2.json", "w") as f:
    json.dump(config, f, indent=2, default=str)

print("Fichiers sauvegardés:")
print(f"  - {model_filename}.zip (modèle {best_name})")
print(f"  - training_config_rl_v2.json")
print(f"  - rl_backtest_v2.png")

In [ ]:
# Télécharger les fichiers
from google.colab import files

print("Téléchargement des fichiers...")
files.download(f'{model_filename}.zip')
files.download('training_config_rl_v2.json')
files.download('rl_backtest_v2.png')

print("\n" + "="*60)
print("INSTRUCTIONS POUR TRADERPRO")
print("="*60)
print(f"""
1. Placez les fichiers dans: backend/app/ml/models/
   - {model_filename}.zip
   - training_config_rl_v2.json

2. Le modèle utilise {best_name} avec observations normalisées

3. Features requises (normalisées entre -1 et 1):
   {config['feature_columns']}

4. Endpoint API: GET /api/rl/action
""")

---
## 12. Test de Prédiction (Simulation Backend)

In [ ]:
def predict_action_v2(model, current_data, position=0, unrealized_pnl=0, steps_in_position=0):
    """
    Prédit l'action à prendre (simulation du backend).
    """
    # Créer les features
    features_df = create_features(current_data)
    
    feature_columns = [
        'returns', 'returns_5d', 'returns_10d', 'volatility_norm',
        'rsi_norm', 'macd_norm', 'bb_norm', 'trend_short', 'trend_long',
        'volume_ratio', 'momentum'
    ]
    
    # Dernière observation
    last_features = features_df[feature_columns].iloc[-1].values
    
    # Normaliser les extras
    position_norm = float(position)
    pnl_norm = np.clip(unrealized_pnl, -0.5, 0.5) * 2
    steps_norm = min(steps_in_position / 30.0, 1.0)
    
    # Construire l'observation
    obs = np.concatenate([
        last_features,
        [position_norm, pnl_norm, steps_norm]
    ]).astype(np.float32)
    
    obs = np.clip(obs, -1.0, 1.0)
    obs = np.nan_to_num(obs, nan=0.0)
    
    # Prédire
    action, _ = model.predict(obs, deterministic=True)
    
    action_names = ["HOLD", "BUY", "SELL"]
    
    return {
        "action": action_names[action],
        "action_code": int(action),
        "current_price": float(current_data['close'].iloc[-1]),
        "model_type": best_name,
        "timestamp": datetime.now().isoformat()
    }

# Test
print("Test de prédiction avec données actuelles:")
print("="*50)

# Sans position
result1 = predict_action_v2(best_model, df.tail(60), position=0)
print(f"Sans position:")
print(f"  Action: {result1['action']}")
print(f"  Prix: ${result1['current_price']:,.2f}")

# Avec position profitable
result2 = predict_action_v2(best_model, df.tail(60), position=1, unrealized_pnl=0.05, steps_in_position=5)
print(f"\nAvec position (+5% profit):")
print(f"  Action: {result2['action']}")

# Avec position en perte
result3 = predict_action_v2(best_model, df.tail(60), position=1, unrealized_pnl=-0.03, steps_in_position=10)
print(f"\nAvec position (-3% perte):")
print(f"  Action: {result3['action']}")

---
## Résumé V2

### Améliorations par rapport à V1:
1. **Episodes courts** (60 jours max) → Apprentissage plus rapide
2. **Observations normalisées** [-1, 1] → Stabilité
3. **Reward shaping progressif** → Signal plus clair
4. **Démarrage aléatoire** → Meilleure généralisation
5. **Comparaison DQN vs A2C** → Choix du meilleur

### Pourquoi DQN/A2C au lieu de PPO?
- **DQN**: Optimisé pour actions discrètes (HOLD/BUY/SELL)
- **A2C**: Plus rapide, updates plus fréquents
- **PPO**: Bon mais peut stagner avec épisodes longs

### Fichiers générés:
- `trading_agent_[dqn|a2c]_final.zip`
- `training_config_rl_v2.json`
- `rl_backtest_v2.png`